This notebook trains the tree-based boosting models using the engineered dataset.

Models trained:

• CatBoost
• LightGBM
• XGBoost

Tasks:

• Load processed datasets
• Load selected features
• Train boosting models
• Compare performance
• Save trained models

In [1]:
# =============================================================================
# IMPORT LIBRARIES
# =============================================================================

import json
import pickle
import random
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

import lightgbm as lgb
import xgboost as xgb

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

# =============================================================================
# REPRODUCIBILITY
# =============================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

# =============================================================================
# PROJECT PATHS
# =============================================================================

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "Data"
PROCESSED_DIR = DATA_DIR / "Processed"

MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================================
# LOAD ENGINEERED DATASET
# =============================================================================

dataset_path = PROCESSED_DIR / "engineered_features.parquet"

df = pd.read_parquet(dataset_path)

# =============================================================================
# LOAD SAVED ARTIFACTS
# =============================================================================

with open(MODEL_DIR / "selected_features.pkl", "rb") as f:
    selected_features = pickle.load(f)

with open(MODEL_DIR / "robust_scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

# =============================================================================
# DATA SUMMARY
# =============================================================================

print("=" * 80)
print("Boosting Model Training")
print("=" * 80)

print(f"Dataset Shape      : {df.shape}")
print(f"Selected Features  : {len(selected_features)}")

print("\nLoaded Files")
print("✓ engineered_features.parquet")
print("✓ selected_features.pkl")
print("✓ robust_scaler.pkl")

print("=" * 80)

Boosting Model Training
Dataset Shape      : (3429120, 53)
Selected Features  : 47

Loaded Files
✓ engineered_features.parquet
✓ selected_features.pkl
✓ robust_scaler.pkl


In [2]:
# =============================================================================
# PREPARE TRAINING DATA
# =============================================================================

from sklearn.model_selection import train_test_split
from catboost import Pool
TARGET_COLUMN = "calculated_aqi"
FEATURE_COLUMNS = selected_features.copy()

# =============================================================================
# SELECT FEATURES & TARGET
# =============================================================================

X = df[FEATURE_COLUMNS].copy()
y = df[TARGET_COLUMN].copy()

# =============================================================================
# TRAIN / VALIDATION / TEST SPLIT
# =============================================================================

# 70% Train | 15% Validation | 15% Test

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, shuffle=False)
X_valid, X_test, y_valid, y_test = train_test_split(X_temp,y_temp,test_size=0.50,shuffle=False)

print("=" * 80)
print("Dataset Split")
print("=" * 80)

print(f"Train      : {X_train.shape}")
print(f"Validation : {X_valid.shape}")
print(f"Test       : {X_test.shape}")

# =============================================================================
# IDENTIFY FEATURE TYPES
# =============================================================================

categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

numerical_features = [
    feature
    for feature in FEATURE_COLUMNS
    if feature not in categorical_features]

print("\nFeature Summary")
print("-" * 80)

print(f"Total Features       : {len(FEATURE_COLUMNS)}")
print(f"Numerical Features   : {len(numerical_features)}")
print(f"Categorical Features : {len(categorical_features)}")

# =============================================================================
# CREATE DATASETS
# =============================================================================

cat_train = Pool(X_train,y_train,cat_features=categorical_features)
cat_valid = Pool(X_valid,y_valid,cat_features=categorical_features)
lgb_train = lgb.Dataset(X_train,label=y_train)
lgb_valid = lgb.Dataset(X_valid, label=y_valid)

print("\nBoosting datasets prepared successfully.")

Dataset Split
Train      : (2400384, 47)
Validation : (514368, 47)
Test       : (514368, 47)

Feature Summary
--------------------------------------------------------------------------------
Total Features       : 47
Numerical Features   : 47
Categorical Features : 0

Boosting datasets prepared successfully.


In [3]:
# =============================================================================
# TRAIN TREE-BASED MODELS
# =============================================================================

from sklearn.metrics import (mean_absolute_error,mean_squared_error,r2_score)
tree_results = []

# =============================================================================
# CATBOOST
# =============================================================================

print("=" * 80)
print("Training CatBoost")
print("=" * 80)

catboost_model = CatBoostRegressor(
    iterations=5000,
    learning_rate=0.03,
    depth=8,
    loss_function="MAE",
    eval_metric="MAE",
    random_seed=SEED,
    verbose=200
)

catboost_model.fit(
    cat_train,
    eval_set=cat_valid,
    use_best_model=True,
    early_stopping_rounds=200
)

catboost_model.save_model(MODEL_DIR / "catboost_model.cbm")
cat_valid_pred = catboost_model.predict(X_valid)
cat_test_pred = catboost_model.predict(X_test)
tree_results.append({
    "Model": "CatBoost",
    "Validation MAE": mean_absolute_error(y_valid, cat_valid_pred),
    "Test MAE": mean_absolute_error(y_test, cat_test_pred),
    "Test RMSE": np.sqrt(mean_squared_error(y_test, cat_test_pred)),
    "Test R2": r2_score(y_test, cat_test_pred)
})

# =============================================================================
# LIGHTGBM
# =============================================================================

print("\n" + "=" * 80)
print("Training LightGBM")
print("=" * 80)

lightgbm_model = LGBMRegressor(
    objective="regression_l1",
    n_estimators=5000,
    learning_rate=0.03,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEED,
    n_jobs=-1
)

lightgbm_model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric="l1",
    callbacks=[
        lgb.early_stopping(200),
        lgb.log_evaluation(200)
    ]
)

joblib.dump(lightgbm_model,MODEL_DIR / "lightgbm_model.pkl")
lgb_valid_pred = lightgbm_model.predict(X_valid)
lgb_test_pred = lightgbm_model.predict(X_test)

tree_results.append({
    "Model": "LightGBM",
    "Validation MAE": mean_absolute_error(y_valid, lgb_valid_pred),
    "Test MAE": mean_absolute_error(y_test, lgb_test_pred),
    "Test RMSE": np.sqrt(mean_squared_error(y_test, lgb_test_pred)),
    "Test R2": r2_score(y_test, lgb_test_pred)
})

# =============================================================================
# XGBOOST
# =============================================================================

print("\n" + "=" * 80)
print("Training XGBoost")
print("=" * 80)

xgboost_model = XGBRegressor(
    objective="reg:absoluteerror",
    n_estimators=5000,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEED,
    tree_method="hist",
    enable_categorical=True,
    n_job = -1
)

xgboost_model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=200
)

xgboost_model.save_model(MODEL_DIR / "xgboost_model.json")
xgb_valid_pred = xgboost_model.predict(X_valid)
xgb_test_pred = xgboost_model.predict(X_test)
tree_results.append({
    "Model": "XGBoost",
    "Validation MAE": mean_absolute_error(y_valid, xgb_valid_pred),
    "Test MAE": mean_absolute_error(y_test, xgb_test_pred),
    "Test RMSE": np.sqrt(mean_squared_error(y_test, xgb_test_pred)),
    "Test R2": r2_score(y_test, xgb_test_pred)
})

# =============================================================================
# MODEL COMPARISON
# =============================================================================

tree_results = (pd.DataFrame(tree_results).sort_values("Validation MAE").reset_index(drop=True))
tree_results.to_csv(MODEL_DIR / "tree_model_results.csv", index=False)
best_model = tree_results.iloc[0]["Model"]
with open(MODEL_DIR / "best_tree_model.txt", "w") as f:
    f.write(best_model)

print("\n" + "=" * 80)
print("Tree Model Performance")
print("=" * 80)

display(tree_results)

print(f"\nBest Tree Model : {best_model}")

print("\nModels Saved")
print("✓ catboost_model.cbm")
print("✓ lightgbm_model.pkl")
print("✓ xgboost_model.json")
print("✓ tree_model_results.csv")

Training CatBoost
0:	learn: 59.3425825	test: 52.9697965	best: 52.9697965 (0)	total: 221ms	remaining: 18m 26s
200:	learn: 1.8830901	test: 1.8786808	best: 1.8786808 (200)	total: 25.4s	remaining: 10m 7s
400:	learn: 1.4156549	test: 1.4221275	best: 1.4221275 (400)	total: 50.9s	remaining: 9m 44s
600:	learn: 1.2884655	test: 1.3095194	best: 1.3095194 (600)	total: 1m 16s	remaining: 9m 21s
800:	learn: 1.2279724	test: 1.2572215	best: 1.2572215 (800)	total: 1m 42s	remaining: 8m 57s
1000:	learn: 1.1912451	test: 1.2255943	best: 1.2255943 (1000)	total: 2m 8s	remaining: 8m 32s
1200:	learn: 1.1662480	test: 1.2045176	best: 1.2045176 (1200)	total: 2m 35s	remaining: 8m 11s
1400:	learn: 1.1457021	test: 1.1865562	best: 1.1865541 (1399)	total: 3m 3s	remaining: 7m 50s
1600:	learn: 1.1280437	test: 1.1711546	best: 1.1711546 (1600)	total: 3m 32s	remaining: 7m 30s
1800:	learn: 1.1141065	test: 1.1588958	best: 1.1588958 (1800)	total: 4m 2s	remaining: 7m 10s
2000:	learn: 1.1024762	test: 1.1489413	best: 1.1489413 (20

,Model,Validation MAE,Test MAE,Test RMSE,Test R2
0,CatBoost,1.078974,0.894903,2.743399,0.998324
1,LightGBM,1.182344,0.979749,2.837925,0.998206
2,XGBoost,1.216809,1.015571,2.846779,0.998195



Best Tree Model : CatBoost

Models Saved
✓ catboost_model.cbm
✓ lightgbm_model.pkl
✓ xgboost_model.json
✓ tree_model_results.csv


In [4]:
# =============================================================================
# VERIFY SAVED MODELS & EXPORT TEST PREDICTIONS
# =============================================================================

print("=" * 80)
print("Loading Saved Models")
print("=" * 80)

# -----------------------------------------------------------------------------
# LOAD SAVED MODELS
# -----------------------------------------------------------------------------

catboost_model = CatBoostRegressor()
catboost_model.load_model(MODEL_DIR / "catboost_model.cbm")

lightgbm_model = joblib.load(MODEL_DIR / "lightgbm_model.pkl")

xgboost_model = XGBRegressor()
xgboost_model.load_model(MODEL_DIR / "xgboost_model.json")

print("✓ CatBoost Loaded")
print("✓ LightGBM Loaded")
print("✓ XGBoost Loaded")

# -----------------------------------------------------------------------------
# GENERATE TEST PREDICTIONS
# -----------------------------------------------------------------------------

catboost_predictions = catboost_model.predict(X_test)
lightgbm_predictions = lightgbm_model.predict(X_test)
xgboost_predictions = xgboost_model.predict(X_test)

# -----------------------------------------------------------------------------
# SAVE PREDICTIONS
# -----------------------------------------------------------------------------

prediction_df = pd.DataFrame({
    "Actual_AQI": y_test.values,
    "CatBoost": catboost_predictions,
    "LightGBM": lightgbm_predictions,
    "XGBoost": xgboost_predictions
})

prediction_path = MODEL_DIR / "tree_predictions.csv"

prediction_df.to_csv(
    prediction_path,
    index=False
)

# -----------------------------------------------------------------------------
# SUMMARY
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("Prediction Export Completed")
print("=" * 80)

print(f"Prediction Shape : {prediction_df.shape}")
print(f"Saved File       : {prediction_path.name}")

display(prediction_df.head())

print("\n✓ Tree model predictions saved successfully.")

Loading Saved Models
✓ CatBoost Loaded
✓ LightGBM Loaded
✓ XGBoost Loaded

Prediction Export Completed
Prediction Shape : (514368, 4)
Saved File       : tree_predictions.csv


,Actual_AQI,CatBoost,LightGBM,XGBoost
0,86.812500,86.362002,85.297671,86.032181
1,87.010414,87.327506,86.818281,86.184738
2,86.218750,86.841968,86.568511,86.634842
3,85.843750,86.084308,86.081274,86.301010
4,85.572914,85.184388,85.767608,85.119690



✓ Tree model predictions saved successfully.
